# EMR Assistant — ASR and diarization benchmark

Compares model combinations on **accuracy, speed and resource cost**, using the
project's own metrics so every figure is comparable with the existing report.

Combinations, in the order the supervisor asked for them:

| # | ASR | Diarization |
|---|---|---|
| 0 | Whisper `base.en` | pyannote 3.1 | *(current system — the control)* |
| 1 | Whisper `medium` | pyannote 3.1 |
| 2 | Whisper `medium` | **Sortformer** |
| 3 | **NeMo Parakeet** | pyannote 3.1 |

Plus a noise sweep on the control, since the pipeline has no preprocessing.

**Recorded for every run:** peak GPU VRAM, peak RAM, CPU seconds, model load time,
inference time, real-time factor, word accuracy, speaker accuracy.

The metric functions are imported from `scripts/evaluate_accuracy.py` in the
repository rather than reimplemented, so a number here and a number in the report
were computed by the same code.

> **From the script's own docstring:** do not tune anything against these
> recordings and re-run. That converts a measurement into a fitting exercise.
> Decide what you are reporting before running, and report every run.

**Runtime → Change runtime type → T4 GPU** before starting.

## 1. Testing hardware

This cell answers the "testing hardware" line in the report. Run it first and
keep the output — Colab does not always give you the same GPU.

In [ ]:
!pip install -q psutil

import platform, subprocess, torch, psutil, json

def hardware_report():
    info = {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu_model": "unknown",
        "cpu_cores_physical": psutil.cpu_count(logical=False),
        "cpu_cores_logical": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "gpu": None,
        "gpu_vram_gb": None,
        "cuda": torch.version.cuda,
    }

    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                info["cpu_model"] = line.split(":", 1)[1].strip()
                break
    except Exception:
        pass

    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["gpu_vram_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / 1e9, 1)

    return info

HARDWARE = hardware_report()
print(json.dumps(HARDWARE, indent=2))

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "cpu_cores_physical": 1,
  "cpu_cores_logical": 2,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}


## 2. Repository and dependencies

The evidence recordings are committed to the repo, so cloning gets the audio and
the reference scripts together. Nothing is uploaded by hand.

In [ ]:
# Safe to re-run: after the restart below, this cell runs a second time.
![ -d repo ] || git clone --depth 1 https://github.com/Wajeeha-Kamran/emr-assistant-backend.git repo
%cd /content/repo
!ls docs/evidence docs/evidence/human_distinct

/content/repo
docs/evidence:
consultation_scripts.md  human_distinct  pipeline_clip.wav  soap_heldout.md
demo_clip.wav		 load_clip.wav	 soap_expected.md   synthetic

docs/evidence/human_distinct:
consult_1.wav  consult_3.wav  diarized_output.txt
consult_2.wav  consult_4.wav  README.txt


In [ ]:
!pip install -q openai-whisper "pyannote.audio==4.0.7" soundfile
!pip install -q -U numba

print("installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 27.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 51.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requiremen

In [ ]:
!pip install -q openai-whisper "pyannote.audio==4.0.7" soundfile
!pip install -q -U numba

print("installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.4 MB/s et

> ## Restart the session now, before going further
>
> **Runtime → Restart session**, then come back and run from the very first cell
> again.
>
> This is not optional. The cell above replaced numpy on disk, but the running
> session is still holding the old one in memory. Carrying on without restarting
> produces an `ImportError` about `numpy._core.umath` when pyannote loads, which
> gives no hint that the real cause was the install.
>
> The second pass is quick — everything is cached, and the clone cell skips
> itself.

### Hugging Face token

pyannote's weights are licence-gated. The token authorises the download and
nothing else. Put it in Colab's secrets (key icon, left sidebar) as `HF_TOKEN`
rather than pasting it into a cell.

Licences needed on `pyannote/segmentation-3.0`,
`pyannote/speaker-diarization-3.1` and `pyannote/speaker-diarization-community-1`.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN is empty"
print("token loaded")

token loaded


## 3. The project's own metrics

`word_error_rate` and `speaker_accuracy` come from the repository, unchanged. The
app imports inside `evaluate_accuracy.main()` are never triggered, so this needs
no database and no `.env`.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, audio_duration,
    SCRIPTS_MD, TARGET,
)

scripts = parse_scripts(SCRIPTS_MD)
print(f"parsed {len(scripts)} reference scripts:", sorted(scripts))

parsed 4 reference scripts: [1, 2, 3, 4]


## 4. Measuring cost

`Measured` wraps a block and records what it consumed. Three notes on method,
because these numbers go in a report:

**CPU seconds, not CPU percent.** A percentage depends on when you sampled it.
CPU-seconds is the total work done and is reproducible.

**Peak VRAM, not final VRAM.** What matters for "will this fit on a given card"
is the high-water mark, not what was still allocated at the end.

**Load time separate from inference time.** Loading a model is a one-off cost paid
at startup; inference is paid per consultation. Averaging them together would
flatter the big models and mislead on deployment.

In [ ]:
import time, gc, threading
import torch, psutil

class Measured:
    """Context manager recording VRAM, RAM, CPU and wall time for a block."""

    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        # Sample RSS in the background: peak RAM is invisible if you only look
        # before and after.
        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)

        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False

## 5. The harness

An engine is a function taking a WAV path and returning the pipeline's own shape:
`[{"text": ..., "speaker_role": "DOCTOR"|"PATIENT"}]`.

`benchmark()` loads the engine once (timed separately), then runs every script
(timed individually), and returns one row per script plus a resource summary.

In [ ]:
import pandas as pd

results = []
resource_rows = []

def benchmark(loader, engine, audio_dir, label, scripts=scripts):
    """Load once, run every script, record accuracy and cost."""
    print(f"\n=== {label} — {os.path.basename(audio_dir)} ===")

    with Measured("load") as load:
        loader()
    print(f"  model load: {load.stats['wall_s']}s, "
          f"VRAM {load.stats['vram_peak_gb']}GB")

    rows = []
    for n in sorted(scripts):
        wav = os.path.join(audio_dir, f"consult_{n}.wav")
        if not os.path.exists(wav):
            print(f"  script {n}: missing, skipped")
            continue

        ref_words, ref_spk = [], []
        for speaker, text in scripts[n]:
            w = normalise(text)
            ref_words.extend(w)
            ref_spk.extend([speaker] * len(w))

        with Measured(f"script{n}") as run:
            segments = engine(wav)

        hyp_words, hyp_spk = [], []
        for seg in segments:
            w = normalise(seg["text"])
            hyp_words.extend(w)
            hyp_spk.extend([seg["speaker_role"]] * len(w))

        wacc_nonum = max(0.0, 1 - word_error_rate(
            strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
        correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
        spk = (correct / total * 100) if total else 0.0
        duration = audio_duration(wav)

        rows.append({
            "run": label,
            "audio_set": os.path.basename(audio_dir),
            "script": n,
            "word_acc": round(wacc_nonum, 1),
            "speaker_acc": round(spk, 1),
            "segments": len(segments),
            "ref_turns": len(scripts[n]),
            "audio_s": round(duration, 1),
            "infer_s": run.stats["wall_s"],
            "realtime_x": round(run.stats["wall_s"] / duration, 2) if duration else None,
            "cpu_s": run.stats["cpu_s"],
            "vram_peak_gb": run.stats["vram_peak_gb"],
            "ram_peak_gb": run.stats["ram_peak_gb"],
        })
        print(f"  script {n}: word {wacc_nonum:.1f}%  speaker {spk:.1f}%  "
              f"{run.stats['wall_s']}s  VRAM {run.stats['vram_peak_gb']}GB")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    resource_rows.append({
        "run": label,
        "audio_set": os.path.basename(audio_dir),
        "load_s": load.stats["wall_s"],
        "load_vram_gb": load.stats["vram_peak_gb"],
        "vram_peak_gb": df.vram_peak_gb.max(),
        "ram_peak_gb": df.ram_peak_gb.max(),
        "cpu_s_mean": round(df.cpu_s.mean(), 1),
        "infer_s_mean": round(df.infer_s.mean(), 1),
        "realtime_x_mean": round(df.realtime_x.mean(), 2),
        "word_acc_mean": round(df.word_acc.mean(), 1),
        "speaker_acc_mean": round(df.speaker_acc.mean(), 1),
    })

    print(f"  MEAN  word {df.word_acc.mean():.1f}%  speaker {df.speaker_acc.mean():.1f}%"
          f"  | peak VRAM {df.vram_peak_gb.max()}GB  RT x{df.realtime_x.mean():.2f}")
    results.append(df)
    return df

## 6. Engines

### Assigning speakers

Diarization produces anonymous clusters. The project names the doctor as
**whichever speaker asks more questions** — history taking is question-driven, so
this is a majority vote across the consultation rather than a guess from who
spoke first.

Every engine below uses this same function. Change it and you stop comparing
diarization models and start comparing naming rules.

In [ ]:
import whisper
from pyannote.audio import Pipeline

_models = {}

def load_whisper(name):
    if name not in _models:
        _models[name] = whisper.load_model(name, device="cuda")
    return _models[name]

def load_pyannote():
    if "pyannote" not in _models:
        p = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                     token=os.environ["HF_TOKEN"])
        p.to(torch.device("cuda"))
        _models["pyannote"] = p
    return _models["pyannote"]


def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    """Give each word the speaker whose turn covers it, then merge runs."""
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        # Nothing covers this moment: take the nearest turn rather than dropping
        # the word, which would silently shorten the hypothesis and flatter WER.
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def whisper_pyannote(wav, model_name):
    asr = load_whisper(model_name).transcribe(wav, word_timestamps=True)
    words = [w for seg in asr.get("segments", []) for w in seg.get("words", [])]
    turns = load_pyannote()(wav)
    spans = [(t.start, t.end, spk) for t, _, spk in turns.itertracks(yield_label=True)]
    return words_to_turns(words, spans)

In [ ]:
# --- PATCH: pyannote 4.x returns DiarizeOutput, not Annotation ---
def _to_spans(turns):
    """Pull (start, end, speaker) out of whatever the pipeline returned."""
    # 1. pyannote 3.x: the object IS the Annotation
    if hasattr(turns, "itertracks"):
        src, how = turns, "Annotation (3.x style)"
    else:
        # 2. pyannote 4.x: DiarizeOutput wrapping one or more Annotations
        cand = None
        for attr in ("speaker_diarization", "diarization", "annotation",
                     "exclusive_speaker_diarization"):
            v = getattr(turns, attr, None)
            if v is not None and hasattr(v, "itertracks"):
                cand, how = v, f"DiarizeOutput.{attr}"
                break
        if cand is None:
            # 3. tuple-like: scan the fields
            try:
                for i, v in enumerate(turns):
                    if hasattr(v, "itertracks"):
                        cand, how = v, f"DiarizeOutput[{i}]"
                        break
            except TypeError:
                pass
        if cand is None:
            raise RuntimeError(
                f"No Annotation found on {type(turns).__name__}. "
                f"Attributes: {[a for a in dir(turns) if not a.startswith('_')]}"
            )
        src = cand
    if not getattr(_to_spans, "_reported", False):
        print(f"[diarization output: {how}]")
        _to_spans._reported = True
    return [(t.start, t.end, spk) for t, _, spk in src.itertracks(yield_label=True)]


def whisper_pyannote(wav, model_name):
    asr = load_whisper(model_name).transcribe(wav, word_timestamps=True)
    words = [w for seg in asr.get("segments", []) for w in seg.get("words", [])]
    spans = _to_spans(load_pyannote()(wav))
    return words_to_turns(words, spans)

print("patched")

patched


## Run 0 — the control

Whisper `base.en` + pyannote: exactly what the system uses today.

**Check this before running anything else.** If word accuracy lands near
**86.4%** and speaker accuracy near **77.6%** on `human_distinct`, the notebook is
sound. If it does not, stop and find out why — every later number would inherit
the same fault.

This run also produces the GPU timing figure that Module 8.3 could never measure,
because there was no GPU.

In [ ]:
benchmark(lambda: (load_whisper("base.en"), load_pyannote()),
          lambda w: whisper_pyannote(w, "base.en"),
          "docs/evidence/human_distinct", "0. base.en + pyannote")

benchmark(lambda: None,
          lambda w: whisper_pyannote(w, "base.en"),
          "docs/evidence/synthetic", "0. base.en + pyannote (synthetic)")


=== 0. base.en + pyannote — human_distinct ===
  model load: 0.0s, VRAM 0.34GB
[diarization output: DiarizeOutput.speaker_diarization]
  script 1: word 86.9%  speaker 100.0%  7.96s  VRAM 2.0GB
  script 2: word 94.8%  speaker 12.4%  8.01s  VRAM 2.0GB
  script 3: word 87.8%  speaker 98.1%  7.06s  VRAM 2.0GB
  script 4: word 83.6%  speaker 100.0%  12.65s  VRAM 2.0GB
  MEAN  word 88.3%  speaker 77.6%  | peak VRAM 2.0GB  RT x0.09

=== 0. base.en + pyannote (synthetic) — synthetic ===
  model load: 0.0s, VRAM 0.34GB
  script 1: word 95.3%  speaker 100.0%  9.78s  VRAM 2.0GB
  script 2: word 98.6%  speaker 100.0%  8.54s  VRAM 2.0GB
  script 3: word 93.4%  speaker 99.4%  8.2s  VRAM 2.0GB
  script 4: word 93.3%  speaker 100.0%  13.63s  VRAM 2.0GB
  MEAN  word 95.1%  speaker 99.8%  | peak VRAM 2.0GB  RT x0.10


,run,audio_set,script,word_acc,speaker_acc,segments,ref_turns,audio_s,infer_s,realtime_x,cpu_s,vram_peak_gb,ram_peak_gb
0,0. base.en + pyannote (synthetic),synthetic,1,95.3,100.0,14,16,108.0,9.78,0.09,9.50,2.0,2.64
1,0. base.en + pyannote (synthetic),synthetic,2,98.6,100.0,15,17,87.9,8.54,0.10,8.32,2.0,2.66
2,0. base.en + pyannote (synthetic),synthetic,3,93.4,99.4,16,17,85.6,8.20,0.10,7.95,2.0,2.64
3,0. base.en + pyannote (synthetic),synthetic,4,93.3,100.0,14,16,142.6,13.63,0.10,13.42,2.0,2.70


## Run 1 — Whisper medium

One variable changed. ASR already meets its 85% target, so the question is what
the extra cost buys — and whether better word timings improve **speaker** accuracy
as a side effect, which is the more interesting possibility.

Watch the VRAM column. `medium` is roughly ten times the parameters of `base.en`.

In [ ]:
benchmark(lambda: load_whisper("medium"),
          lambda w: whisper_pyannote(w, "medium"),
          "docs/evidence/human_distinct", "1. medium + pyannote")

NameError: name 'benchmark' is not defined

## Run 2 — Whisper medium + Sortformer

**Scaffold, not a working engine.** NeMo's install and API move between releases
and I could not verify this from outside a running Colab, so treat it as a
starting point to correct rather than code to trust.

Only the diarization call changes. Keep `words_to_turns` and `assign_roles`
exactly as they are — that is what isolates the variable.

In [ ]:
# !pip install -q "nemo_toolkit[asr]"

def load_sortformer():
    """
    Something like:

        from nemo.collections.asr.models import SortformerEncLabelModel
        _models["sortformer"] = SortformerEncLabelModel.from_pretrained(
            "nvidia/diar_sortformer_4spk-v1").to("cuda").eval()

    Check the model card for the current class name and checkpoint.
    """
    raise NotImplementedError("Verify against the NeMo docs for the installed version.")


def whisper_sortformer(wav, model_name="medium"):
    """
    Same three steps as whisper_pyannote, with step 2 swapped:

      1. Whisper transcribe, word_timestamps=True          (unchanged)
      2. Sortformer -> spans of (start, end, speaker_id)   (the only difference)
      3. words_to_turns(words, spans)                      (unchanged)

    Sortformer returns per-frame speaker activity rather than turn objects, so
    step 2 needs converting to (start, end, label) spans before step 3 will take
    it.
    """
    raise NotImplementedError

# benchmark(load_sortformer, whisper_sortformer,
#           "docs/evidence/human_distinct", "2. medium + sortformer")

## Run 3 — NeMo Parakeet

Also a scaffold. Parakeet is fast and accurate on English, but the important
detail for this project is whether it gives **word-level timestamps** — without
them, step 3 has nothing to map onto speaker turns and the diarization half of
the comparison collapses.

Check that first. If it cannot, say so in the report; that is a finding about
pipeline compatibility, not a failure to test.

In [ ]:
def load_parakeet():
    """
        from nemo.collections.asr.models import ASRModel
        _models["parakeet"] = ASRModel.from_pretrained(
            "nvidia/parakeet-tdt-0.6b-v2").to("cuda").eval()
    """
    raise NotImplementedError("Verify the current checkpoint name on the model card.")


def parakeet_pyannote(wav):
    """
    Needs word timestamps. In recent NeMo that is roughly:

        out = _models["parakeet"].transcribe([wav], timestamps=True)
        words = [{"word": w["word"], "start": w["start"], "end": w["end"]}
                 for w in out[0].timestamp["word"]]

    then the same words_to_turns(words, spans) as everything else.
    """
    raise NotImplementedError

# benchmark(load_parakeet, parakeet_pyannote,
#           "docs/evidence/human_distinct", "3. parakeet + pyannote")

## Run 4 — noise

There is no audio preprocessing in the pipeline: recordings go to Whisper exactly
as captured. This measures what that costs.

Gaussian noise at three signal-to-noise ratios — 20 dB is a quiet room, 10 dB a
busy clinic, 5 dB is bad. Fixed seed, so reruns are repeatable. Noisy copies go to
`/content`, leaving the evidence files untouched.

In [ ]:
import numpy as np, soundfile as sf

def add_noise(src_dir, dst_dir, snr_db, seed=0):
    os.makedirs(dst_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    for name in sorted(os.listdir(src_dir)):
        if not (name.startswith("consult_") and name.endswith(".wav")):
            continue
        audio, sr = sf.read(os.path.join(src_dir, name))
        noise_power = np.mean(audio ** 2) / (10 ** (snr_db / 10))
        noisy = np.clip(audio + rng.normal(0, np.sqrt(noise_power), audio.shape), -1, 1)
        sf.write(os.path.join(dst_dir, name), noisy, sr, subtype="PCM_16")
    return dst_dir

for snr in (20, 10, 5):
    add_noise("docs/evidence/human_distinct", f"/content/noise_{snr}db", snr)
print("noisy copies written")

In [ ]:
for snr in (20, 10, 5):
    benchmark(lambda: None,
              lambda w: whisper_pyannote(w, "base.en"),
              f"/content/noise_{snr}db", f"4. base.en + pyannote @ {snr}dB SNR")

## Results

Two tables. The first is the comparison the report needs; the second is per-script
detail for the appendix.

In [ ]:
import pandas as pd

detail = pd.concat([d for d in results if not d.empty], ignore_index=True)
comparison = pd.DataFrame(resource_rows)

print("HARDWARE")
for k, v in HARDWARE.items():
    print(f"  {k}: {v}")
print()

display(comparison)

detail.to_csv("/content/benchmark_detail.csv", index=False)
comparison.to_csv("/content/benchmark_comparison.csv", index=False)
print("\nwritten to /content/benchmark_detail.csv and /content/benchmark_comparison.csv")

### Recommended hardware

Derived from what was measured, not guessed. The rule below is peak VRAM plus
roughly 40% headroom, because a card sized exactly to the peak will fail the first
time a consultation runs slightly long.

In [ ]:
for row in resource_rows:
    vram = row["vram_peak_gb"] or 0
    needed = round(vram * 1.4, 1)
    rt = row["realtime_x_mean"]
    print(f"{row['run']}")
    print(f"   peak VRAM {vram} GB  ->  recommend a card with at least {needed} GB")
    print(f"   peak RAM  {row['ram_peak_gb']} GB")
    print(f"   real-time factor x{rt}  ->  a 10-minute consultation takes "
          f"about {round(rt * 600 / 60, 1)} minutes")
    print()

In [ ]:
# ============================================================
# CELL 1 of 2 — SETUP. Paste this whole thing into one new cell and run it.
# Takes about a minute. Safe to run again any time Colab restarts.
# ============================================================

import os, sys, subprocess

# --- repo -----------------------------------------------------------------
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Wajeeha-Kamran/emr-assistant-backend.git",
                    "/content/repo"], check=True)
os.chdir("/content/repo")
sys.path.insert(0, os.getcwd())

# --- packages (skipped if already installed) -------------------------------
try:
    import whisper, pyannote.audio, soundfile, psutil  # noqa
    print("packages already present")
except ImportError:
    print("installing packages — if this line appears, RESTART THE SESSION")
    print("afterwards (Runtime > Restart session) and run this cell again.")
    subprocess.run("pip install -q openai-whisper 'pyannote.audio==4.0.7' "
                   "soundfile psutil && pip install -q -U numba",
                   shell=True, check=True)
    raise SystemExit("Installed. Now: Runtime > Restart session, then re-run this cell.")

# --- hardware --------------------------------------------------------------
import platform, json, torch, psutil

def hardware_report():
    info = {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu_model": "unknown",
        "cpu_cores_physical": psutil.cpu_count(logical=False),
        "cpu_cores_logical": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "gpu": None, "gpu_vram_gb": None, "cuda": torch.version.cuda,
    }
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                info["cpu_model"] = line.split(":", 1)[1].strip()
                break
    except Exception:
        pass
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["gpu_vram_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    return info

HARDWARE = hardware_report()
assert HARDWARE["gpu"], "No GPU. Runtime > Change runtime type > T4 GPU, then re-run."

# --- HF token --------------------------------------------------------------
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN secret is empty"

# --- the project's own metrics --------------------------------------------
from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, audio_duration, SCRIPTS_MD, TARGET,
)
scripts = parse_scripts(SCRIPTS_MD)

# --- cost measurement ------------------------------------------------------
import time, gc, threading

class Measured:
    """Records VRAM, RAM, CPU and wall time for a block."""

    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)
        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False

# --- harness ---------------------------------------------------------------
import pandas as pd

results = []
resource_rows = []

def benchmark(loader, engine, audio_dir, label, scripts=scripts):
    print(f"\n=== {label} — {os.path.basename(audio_dir)} ===")
    with Measured("load") as load:
        loader()
    print(f"  model load: {load.stats['wall_s']}s, VRAM {load.stats['vram_peak_gb']}GB")

    rows = []
    for n in sorted(scripts):
        wav = os.path.join(audio_dir, f"consult_{n}.wav")
        if not os.path.exists(wav):
            print(f"  script {n}: missing, skipped")
            continue

        ref_words, ref_spk = [], []
        for speaker, text in scripts[n]:
            w = normalise(text)
            ref_words.extend(w)
            ref_spk.extend([speaker] * len(w))

        with Measured(f"script{n}") as run:
            segments = engine(wav)

        hyp_words, hyp_spk = [], []
        for seg in segments:
            w = normalise(seg["text"])
            hyp_words.extend(w)
            hyp_spk.extend([seg["speaker_role"]] * len(w))

        wacc = max(0.0, 1 - word_error_rate(
            strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
        correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
        spk = (correct / total * 100) if total else 0.0
        duration = audio_duration(wav)

        rows.append({
            "run": label, "audio_set": os.path.basename(audio_dir), "script": n,
            "word_acc": round(wacc, 1), "speaker_acc": round(spk, 1),
            "segments": len(segments), "ref_turns": len(scripts[n]),
            "audio_s": round(duration, 1), "infer_s": run.stats["wall_s"],
            "realtime_x": round(run.stats["wall_s"] / duration, 2) if duration else None,
            "cpu_s": run.stats["cpu_s"],
            "vram_peak_gb": run.stats["vram_peak_gb"],
            "ram_peak_gb": run.stats["ram_peak_gb"],
        })
        print(f"  script {n}: word {wacc:.1f}%  speaker {spk:.1f}%  "
              f"{run.stats['wall_s']}s  VRAM {run.stats['vram_peak_gb']}GB")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    resource_rows.append({
        "run": label, "audio_set": os.path.basename(audio_dir),
        "load_s": load.stats["wall_s"], "load_vram_gb": load.stats["vram_peak_gb"],
        "vram_peak_gb": df.vram_peak_gb.max(), "ram_peak_gb": df.ram_peak_gb.max(),
        "cpu_s_mean": round(df.cpu_s.mean(), 1),
        "infer_s_mean": round(df.infer_s.mean(), 1),
        "realtime_x_mean": round(df.realtime_x.mean(), 2),
        "word_acc_mean": round(df.word_acc.mean(), 1),
        "speaker_acc_mean": round(df.speaker_acc.mean(), 1),
    })
    print(f"  MEAN  word {df.word_acc.mean():.1f}%  speaker {df.speaker_acc.mean():.1f}%"
          f"  | peak VRAM {df.vram_peak_gb.max()}GB  RT x{df.realtime_x.mean():.2f}")
    results.append(df)
    return df

# --- engines ---------------------------------------------------------------
import whisper
from pyannote.audio import Pipeline

_models = {}

def load_whisper(name):
    if name not in _models:
        _models[name] = whisper.load_model(name, device="cuda")
    return _models[name]

def load_pyannote():
    if "pyannote" not in _models:
        p = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                     token=os.environ["HF_TOKEN"])
        p.to(torch.device("cuda"))
        _models["pyannote"] = p
    return _models["pyannote"]

def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments

def words_to_turns(words, spans):
    """Give each word the speaker whose turn covers it, then merge runs."""
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)

def _to_spans(turns):
    """pyannote 4.x returns DiarizeOutput; 3.x returned the Annotation itself."""
    if hasattr(turns, "itertracks"):
        src = turns
    else:
        src = None
        for attr in ("speaker_diarization", "diarization", "annotation",
                     "exclusive_speaker_diarization"):
            v = getattr(turns, attr, None)
            if v is not None and hasattr(v, "itertracks"):
                src = v
                break
        if src is None:
            raise RuntimeError(
                f"No Annotation on {type(turns).__name__}: "
                f"{[a for a in dir(turns) if not a.startswith('_')]}")
    return [(t.start, t.end, spk) for t, _, spk in src.itertracks(yield_label=True)]

def whisper_pyannote(wav, model_name):
    asr = load_whisper(model_name).transcribe(wav, word_timestamps=True)
    words = [w for seg in asr.get("segments", []) for w in seg.get("words", [])]
    return words_to_turns(words, _to_spans(load_pyannote()(wav)))

# --- checkpointing (so a Colab restart never costs you a run) ---------------
CKPT_DETAIL = "/content/ckpt_detail.csv"
CKPT_RESOURCE = "/content/ckpt_resource.csv"

def save_checkpoint():
    if results:
        pd.concat([x for x in results if not x.empty],
                  ignore_index=True).to_csv(CKPT_DETAIL, index=False)
    if resource_rows:
        pd.DataFrame(resource_rows).to_csv(CKPT_RESOURCE, index=False)
    print(f"checkpoint saved ({len(resource_rows)} runs)")

def load_checkpoint():
    """Put previously saved runs back after a restart."""
    if os.path.exists(CKPT_DETAIL):
        prev = pd.read_csv(CKPT_DETAIL)
        results.clear(); results.append(prev)
        print(f"restored: {sorted(prev.run.unique())}")
    if os.path.exists(CKPT_RESOURCE):
        resource_rows.clear()
        resource_rows.extend(pd.read_csv(CKPT_RESOURCE).to_dict("records"))

load_checkpoint()

print()
print("SETUP OK")
print(json.dumps(HARDWARE, indent=2))
print(f"{len(scripts)} reference scripts: {sorted(scripts)}")
print("Now run CELL 2.")


packages already present
restored: ['0. base.en + pyannote', '0. base.en + pyannote (synthetic)', '1. medium + pyannote', '2. medium + sortformer', '4. base.en + pyannote @ 10dB SNR', '4. base.en + pyannote @ 20dB SNR', '4. base.en + pyannote @ 5dB SNR']

SETUP OK
{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "cpu_cores_physical": 1,
  "cpu_cores_logical": 2,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}
4 reference scripts: [1, 2, 3, 4]
Now run CELL 2.


In [ ]:
# ============================================================
# CELL 2 of 2 — THE WHISPER RUNS. Paste into a second new cell and run it.
# Takes roughly 15-25 minutes. It saves after every run, so a disconnect
# part-way through only costs you the run that was in progress.
# ============================================================

# --- Run 0: the control (base.en + pyannote), human audio then synthetic ---
benchmark(lambda: (load_whisper("base.en"), load_pyannote()),
          lambda w: whisper_pyannote(w, "base.en"),
          "docs/evidence/human_distinct", "0. base.en + pyannote")
save_checkpoint()

benchmark(lambda: None,
          lambda w: whisper_pyannote(w, "base.en"),
          "docs/evidence/synthetic", "0. base.en + pyannote (synthetic)")
save_checkpoint()

# --- Run 1: Whisper medium. Downloads ~1.5GB the first time. ---------------
benchmark(lambda: load_whisper("medium"),
          lambda w: whisper_pyannote(w, "medium"),
          "docs/evidence/human_distinct", "1. medium + pyannote")
save_checkpoint()

# --- Run 4: noise sweep on the control -------------------------------------
import numpy as np, soundfile as sf

def add_noise(src_dir, dst_dir, snr_db, seed=0):
    os.makedirs(dst_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    for name in sorted(os.listdir(src_dir)):
        if not (name.startswith("consult_") and name.endswith(".wav")):
            continue
        audio, sr = sf.read(os.path.join(src_dir, name))
        noise_power = np.mean(audio ** 2) / (10 ** (snr_db / 10))
        noisy = np.clip(audio + rng.normal(0, np.sqrt(noise_power), audio.shape), -1, 1)
        sf.write(os.path.join(dst_dir, name), noisy, sr, subtype="PCM_16")
    return dst_dir

for snr in (20, 10, 5):
    add_noise("docs/evidence/human_distinct", f"/content/noise_{snr}db", snr)

for snr in (20, 10, 5):
    benchmark(lambda: None,
              lambda w: whisper_pyannote(w, "base.en"),
              f"/content/noise_{snr}db", f"4. base.en + pyannote @ {snr}dB SNR")
    save_checkpoint()

# --- what we have so far ---------------------------------------------------
print()
print("=" * 70)
print("RESULTS SO FAR")
print("=" * 70)
display(pd.DataFrame(resource_rows))

pd.concat([d for d in results if not d.empty],
          ignore_index=True).to_csv("/content/benchmark_detail.csv", index=False)
pd.DataFrame(resource_rows).to_csv("/content/benchmark_comparison.csv", index=False)
print("\nWritten: /content/benchmark_detail.csv and /content/benchmark_comparison.csv")
print("All the Whisper work is done. Send me these two files.")



=== 0. base.en + pyannote — human_distinct ===


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 184MiB/s]


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

  script 1: word 86.9%  speaker 100.0%  15.58s  VRAM 2.0GB
  script 2: word 94.8%  speaker 12.4%  7.53s  VRAM 2.0GB
  script 3: word 87.8%  speaker 98.1%  8.29s  VRAM 2.0GB
  script 4: word 83.6%  speaker 100.0%  12.36s  VRAM 2.0GB
  MEAN  word 88.3%  speaker 77.6%  | peak VRAM 2.0GB  RT x0.11
checkpoint saved (1 runs)

=== 0. base.en + pyannote (synthetic) — synthetic ===
  model load: 0.0s, VRAM 0.34GB
  script 1: word 95.3%  speaker 100.0%  9.87s  VRAM 2.0GB
  script 2: word 98.6%  speaker 100.0%  8.26s  VRAM 2.0GB
  script 3: word 93.4%  speaker 99.4%  8.21s  VRAM 2.0GB
  script 4: word 93.3%  speaker 100.0%  13.87s  VRAM 2.0GB
  MEAN  word 95.1%  speaker 99.8%  | peak VRAM 2.0GB  RT x0.10
checkpoint saved (2 runs)

=== 1. medium + pyannote — human_distinct ===


100%|██████████████████████████████████████| 1.42G/1.42G [00:07<00:00, 213MiB/s]


  model load: 22.33s, VRAM 4.92GB
  script 1: word 90.6%  speaker 100.0%  17.54s  VRAM 5.06GB
  script 2: word 92.0%  speaker 13.2%  15.9s  VRAM 5.06GB
  script 3: word 89.0%  speaker 98.1%  14.87s  VRAM 5.06GB
  script 4: word 84.5%  speaker 100.0%  26.16s  VRAM 5.06GB
  MEAN  word 89.0%  speaker 77.8%  | peak VRAM 5.06GB  RT x0.19
checkpoint saved (3 runs)

=== 4. base.en + pyannote @ 20dB SNR — noise_20db ===
  model load: 0.0s, VRAM 3.39GB
  script 1: word 87.3%  speaker 100.0%  7.93s  VRAM 5.06GB
  script 2: word 93.0%  speaker 96.5%  7.33s  VRAM 5.06GB
  script 3: word 87.3%  speaker 100.0%  6.92s  VRAM 5.06GB
  script 4: word 83.3%  speaker 100.0%  12.51s  VRAM 5.06GB
  MEAN  word 87.7%  speaker 99.1%  | peak VRAM 5.06GB  RT x0.09
checkpoint saved (4 runs)

=== 4. base.en + pyannote @ 10dB SNR — noise_10db ===
  model load: 0.0s, VRAM 3.39GB
  script 1: word 84.0%  speaker 100.0%  8.0s  VRAM 5.06GB
  script 2: word 86.4%  speaker 96.2%  7.59s  VRAM 5.06GB
  script 3: word 81.2% 

,run,audio_set,load_s,load_vram_gb,vram_peak_gb,ram_peak_gb,cpu_s_mean,infer_s_mean,realtime_x_mean,word_acc_mean,speaker_acc_mean
0,0. base.en + pyannote,human_distinct,12.28,0.44,2.00,2.65,9.8,10.9,0.12,88.3,77.6
1,0. base.en + pyannote (synthetic),synthetic,0.00,0.34,2.00,2.69,9.7,10.1,0.10,95.1,99.8
2,1. medium + pyannote,human_distinct,22.33,4.92,5.06,6.22,18.3,18.6,0.19,89.0,77.8
3,4. base.en + pyannote @ 20dB SNR,noise_20db,0.00,3.39,5.06,6.22,8.5,8.7,0.09,87.7,99.1
4,4. base.en + pyannote @ 10dB SNR,noise_10db,0.00,3.39,5.06,6.22,8.6,8.7,0.09,82.9,99.0
5,4. base.en + pyannote @ 5dB SNR,noise_5db,0.00,3.39,5.06,6.22,8.5,8.7,0.09,78.3,99.4



Written: /content/benchmark_detail.csv and /content/benchmark_comparison.csv
All the Whisper work is done. Send me these two files.


In [ ]:
# ============================================================
# CELL 3 — INSTALL NeMo. Paste into a new cell and run it.
# Takes 5-10 minutes and prints a lot of dependency warnings. Warnings are fine.
# When it finishes: Runtime > Restart session, then run CELL 1 again, then CELL 4.
# ============================================================

!apt-get -qq update > /dev/null && apt-get -qq install -y libsndfile1 ffmpeg > /dev/null
!pip install -q Cython packaging
!pip install -q "nemo_toolkit[asr]"
!pip install -q -U numba

print()
print("=" * 60)
print("NeMo installed.")
print("NEXT: Runtime > Restart session")
print("THEN: run CELL 1 again (it will restore your saved results)")
print("THEN: run CELL 4")
print("=" * 60)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL 4 — SORTFORMER AND PARAKEET. Paste into a new cell and run it.
# Run CELL 1 first (after the restart) so your earlier results are restored.
# Takes 15-30 minutes including model downloads.
# ============================================================

import os, torch, soundfile as sf

# --- Sortformer and Parakeet both want 16 kHz mono -------------------------
_SF_CACHE = {}

def _mono16k(wav):
    """A 16 kHz mono copy under /content. The evidence files are left alone."""
    if wav in _SF_CACHE:
        return _SF_CACHE[wav]
    audio, sr = sf.read(wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio.astype("float32"), orig_sr=sr, target_sr=16000)
    os.makedirs("/content/sf16k", exist_ok=True)
    out = "/content/sf16k/" + os.path.basename(wav)
    sf.write(out, audio, 16000, subtype="PCM_16")
    _SF_CACHE[wav] = out
    return out


# --- Sortformer (replaces pyannote; Whisper medium still does the words) ---
def load_sortformer():
    if "sortformer" not in _models:
        from nemo.collections.asr.models import SortformerEncLabelModel
        m = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
        m.eval()
        if torch.cuda.is_available():
            m = m.cuda()
        _models["sortformer"] = m
    return _models["sortformer"]


def _sortformer_spans(wav):
    """diarize() returns, per file, segments as 'start end speaker' strings."""
    pred = load_sortformer().diarize(audio=[_mono16k(wav)], batch_size=1)
    segs = pred[0] if isinstance(pred, (list, tuple)) and pred else pred
    spans = []
    for s in segs:
        if isinstance(s, str):
            p = s.replace(",", " ").split()
            spans.append((float(p[0]), float(p[1]), str(p[2])))
        elif isinstance(s, (list, tuple)) and len(s) >= 3:
            spans.append((float(s[0]), float(s[1]), str(s[2])))
        else:
            raise RuntimeError(f"unexpected Sortformer segment: {type(s)} {s!r}")
    if not getattr(_sortformer_spans, "_seen", False):
        print(f"  [sortformer: {len(spans)} spans, first={spans[0] if spans else None}]")
        _sortformer_spans._seen = True
    return spans


def whisper_sortformer(wav):
    asr = load_whisper("medium").transcribe(wav, word_timestamps=True)
    words = [w for seg in asr.get("segments", []) for w in seg.get("words", [])]
    return words_to_turns(words, _sortformer_spans(wav))


benchmark(lambda: (load_whisper("medium"), load_sortformer()),
          whisper_sortformer,
          "docs/evidence/human_distinct", "2. medium + sortformer")
save_checkpoint()


# --- Parakeet (replaces Whisper; pyannote still does the speakers) ---------
def load_parakeet():
    if "parakeet" not in _models:
        import nemo.collections.asr as nemo_asr
        m = nemo_asr.models.ASRModel.from_pretrained(
            model_name="nvidia/parakeet-tdt-0.6b-v2")
        m.eval()
        _models["parakeet"] = m
    return _models["parakeet"]


def parakeet_words(wav):
    """Parakeet's tokens have no leading space; Whisper's do, and words_to_turns
    concatenates them directly. Without the space the words would glue together
    and the word-accuracy figure would be meaningless."""
    out = load_parakeet().transcribe([_mono16k(wav)], timestamps=True)
    stamps = out[0].timestamp["word"]
    words = [{"start": float(s["start"]), "end": float(s["end"]),
              "word": " " + str(s["word"]).strip()} for s in stamps]
    if not getattr(parakeet_words, "_seen", False):
        print(f"  [parakeet: {len(words)} words, first={words[0] if words else None}]")
        parakeet_words._seen = True
    return words


def parakeet_pyannote(wav):
    return words_to_turns(parakeet_words(wav), _to_spans(load_pyannote()(wav)))


benchmark(lambda: (load_parakeet(), load_pyannote()),
          parakeet_pyannote,
          "docs/evidence/human_distinct", "3. parakeet + pyannote")
save_checkpoint()


# --- final export ----------------------------------------------------------
import pandas as pd

print()
print("=" * 70)
print("FULL RESULTS")
print("=" * 70)
display(pd.DataFrame(resource_rows))

pd.concat([d for d in results if not d.empty],
          ignore_index=True).to_csv("/content/benchmark_detail.csv", index=False)
pd.DataFrame(resource_rows).to_csv("/content/benchmark_comparison.csv", index=False)
print("\nWritten: /content/benchmark_detail.csv and /content/benchmark_comparison.csv")
print("Download both and send them to me.")



=== 2. medium + sortformer — human_distinct ===


diar_sortformer_4spk-v1.nemo: reconstructing file:   0%|          |  0.00B /  493MB            

diar_sortformer_4spk-v1.nemo: downloading bytes:           |  0.00B            

[NeMo W 2026-08-19 12:35:41 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-08-19 12:35:41 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-08-19 12:35:43 save_restore_connector:287] Model SortformerEncLabelModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--diar_sortformer_4spk-v1/snapshots/9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08/diar_sortformer_4spk-v1.nemo.
  model load: 37.93s, VRAM 4.58GB
[NeMo I 2026-08-19 12:35:59 vad_utils:89] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


...

[NeMo W 2026-08-19 12:37:53 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-08-19 12:37:53 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    use_

[NeMo I 2026-08-19 12:37:58 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 12:37:58 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 12:37:58 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-08-19 12:38:10 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/ae9ad07059c7c739ffaf932226a8fe64ae2620b0/parakeet-tdt-0.6b-v2.nemo.
  model load: 86.72s, VRAM 8.6GB
[NeMo I 2026-08-19 12:38:13 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,            

[NeMo W 2026-08-19 12:38:13 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 12:38:13 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:01,  1.63s/it]
[NeMo W 2026-08-19 12:38:15 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
    It can be re-enabled by calling
       >>> import torch
       >>> torch.backends.cuda.matmul.allow_tf32 = True
       >>> torch.backen

  [parakeet: 215 words, first={'start': 0.96, 'end': 1.44, 'word': ' Good'}]
  script 1: word 90.6%  speaker 100.0%  7.31s  VRAM 7.9GB
[NeMo I 2026-08-19 12:38:21 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']


[NeMo W 2026-08-19 12:38:21 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-19 12:38:21 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s]

In [ ]:
# Report what we have, then put numba back where numba-cuda can work with it.
import importlib.metadata as md
for p in ("numba", "numba-cuda", "numpy", "torch"):
    try:
        print(f"{p:12} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:12} not installed")

!pip install -q "numba==0.61.2"

print()
print("=" * 60)
print("NEXT: Runtime > Restart session")
print("THEN: run CELL 1, then CELL 4")
print("=" * 60)

numba        0.67.0
numba-cuda   0.22.2
numpy        2.5.2
torch        2.11.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 100.0 MB/s eta 0:00:00

NEXT: Runtime > Restart session
THEN: run CELL 1, then CELL 4


In [ ]:
!pip install -q "numba==0.67.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.67.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.


In [ ]:
# ============================================================
# CELL 5 — DUMP THE WHISPER/PYANNOTE SIDE TO A FILE.
#
# Run this in your CURRENT notebook, after:
#   1. a new cell with:  !pip install -q "numba==0.67.0"
#   2. Runtime > Restart session
#   3. CELL 1
#
# It records, for every script: the Whisper medium words, the pyannote speaker
# spans, and how long each stage took. Sortformer and Parakeet then run in a
# clean notebook that never has to import whisper or pyannote at all.
# ============================================================

import json

AUDIO_DIR = "docs/evidence/human_distinct"

load_whisper("medium")
load_pyannote()

dump = {"hardware": HARDWARE, "audio_dir": AUDIO_DIR, "scripts": {}}

for n in sorted(scripts):
    wav = f"{AUDIO_DIR}/consult_{n}.wav"

    with Measured(f"whisper{n}") as mw:
        asr = load_whisper("medium").transcribe(wav, word_timestamps=True)
    words = [{"start": float(w["start"]), "end": float(w["end"]), "word": w["word"]}
             for seg in asr.get("segments", []) for w in seg.get("words", [])]

    with Measured(f"pyannote{n}") as mp:
        spans = [[float(a), float(b), str(c)] for a, b, c in
                 _to_spans(load_pyannote()(wav))]

    dump["scripts"][str(n)] = {
        "audio_s": audio_duration(wav),
        "whisper_medium_words": words,
        "whisper_medium_stats": mw.stats,
        "pyannote_spans": spans,
        "pyannote_stats": mp.stats,
    }
    print(f"script {n}: {len(words)} words ({mw.stats['wall_s']}s), "
          f"{len(spans)} spans ({mp.stats['wall_s']}s)")

with open("/content/handoff.json", "w") as f:
    json.dump(dump, f)

import os
print(f"\nWritten /content/handoff.json "
      f"({os.path.getsize('/content/handoff.json')/1e6:.1f} MB)")
print("Download it, along with benchmark_detail.csv and benchmark_comparison.csv.")


AttributeError: module 'numba.cuda.types' has no attribute 'NPDatetime'

### Before writing any of this up

- **State the audio set with every figure.** `human_distinct` and `synthetic` are
  not comparable, and the report already says so.
- **Report every run, not the best one.**
- **Say what a win costs.** A model that gains two points of accuracy for three
  times the VRAM is a different recommendation from one that gains two points for
  free.
- **A null result is a result.** "Three alternatives were tested and the original
  held up" is a stronger finding than most people expect, and it is only
  available to you if you say so before you look.